# 1D CNN-A — Visualise: Contact Sheet

Runs the full data pipeline and renders `N_SAMPLE` sliding windows as a
**contact sheet** — every 64 × 14-pixel window placed side by side at `SCALE`×
magnification, `GRID_COLS_A` columns per row, with a 1-pixel gap between blocks.

Each block is one 64-bar window of TSLA trading. Columns are the 14 scaled
features; rows are time bars. Dark = low value, bright = high value.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from config import Config
from data import (
    load_bars, clean_data, add_features, drop_feature_nans,
    scale_features, make_windows, filter_gap_windows,
)
from visualise import prepare_sample, draw_contact_sheet

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    import httpx  # only needed when FETCH_DATA = True
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
# Load raw OHLCV bars from the CSV file.
df = load_bars(DATA_DIR, SYMBOL, TIMEFRAME, MAX_BARS)

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
# Remove duplicate timestamps and rows with missing values.
df = clean_data(df)

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# Calculate 14 technical indicator columns (EMAs, MACD, candle shape, returns, volume ratio).
df = add_features(df)

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
# Drop the warm-up rows where EMAs and rolling means don't have enough history yet.
df = drop_feature_nans(df)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
# Normalise every feature column so they all sit in a similar numeric range.
# RobustScaler uses the median and IQR — better than mean/std for financial data with outliers.
df, scaler = scale_features(df, feature_cols)

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
# Slice the time series into overlapping WINDOW_SIZE-bar windows.
# Each window is one training sample for the CNN.
X_raw = make_windows(df, feature_cols, WINDOW_SIZE)
n_features = len(feature_cols)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
# Remove windows that span overnight or weekend gaps.
# Such windows would teach the model noise rather than real patterns.
X_clean, valid_mask = filter_gap_windows(X_raw, df, WINDOW_SIZE)

## 9. Visualise Windows as Greyscale Tiles

Before training, it's worth seeing what the windowed data actually *looks like*.
Each of the 156,781 clean windows is a 64-bar × 14-feature matrix of scaled
floats. Rendering them as greyscale images (bright = high value, dark = low)
lets us visually inspect the dataset for structure, gaps, outliers, and feature
behaviour at a scale no table or line-chart can match.

Three views are provided — all use the same global [0 → 255] normalisation so
brightness is directly comparable across windows and across views:

| View | What you see | Best for |
|------|-------------|----------|
| A — Contact sheet | Each window at full 64×14 px resolution, tiled in a grid | Inspecting individual window texture and feature patterns |
| B — Heatmap strip | Each window flattened to one row, all windows stacked vertically | Spotting dataset-wide trends, session rhythms, and drift over time |
| C — Thumbnail grid | Each window compressed to 10×10 px, tiled densely | Getting a high-level overview of brightness/contrast across the whole sample |

Adjust `N_SAMPLE` in the setup cell below to control how many windows are rendered.

In [ ]:
# Take the first N_SAMPLE windows and convert pixel values to uint8 [0–255].
# Percentile clipping (p2/p98) prevents outliers from washing out the image.
sample_u8, N_SAMPLE, lo, hi = prepare_sample(X_clean, N_SAMPLE)

### View A — Contact Sheet (natural resolution: 64 × 14 px per window)

Each window is rendered at its natural shape: **64 pixels tall** (one pixel row
per bar) and **14 pixels wide** (one pixel column per feature). No information
is lost — every pixel represents exactly one bar/feature value.

Windows are tiled left-to-right across the canvas, wrapping into new rows, like
a filmstrip or contact sheet. A 1-pixel gap separates each block so individual
windows are distinguishable.

**What to look for:**
- **Vertical banding** within a block: a feature column that is consistently
  bright or dark across all 64 bars — likely an EMA or price feature at an extreme.
- **Horizontal banding** within a block: a single bar (row) that stands out
  across all 14 features simultaneously — could be a volume spike or large candle.
- **Blocks that look uniform / flat**: windows where all features stayed near the
  median (scaled to mid-grey ~128). Common during low-volatility periods.
- **Blocks that look noisy**: highly varied pixel values — active, volatile conditions.
- **Repeating texture patterns** across many blocks: candidate patterns the
  autoencoder should learn to encode and reconstruct.

In [ ]:
# Draw View A: each window as a small pixel block, tiled in a grid.
fig = draw_contact_sheet(sample_u8, SCALE, GRID_COLS_A, GAP_PX, feature_cols, WINDOW_SIZE)
plt.show()